# Response Quality Evaluation — BLEU, ROUGE-L, BERTScore
Measures text generation quality against reference answers.

In [ ]:
# pip install rouge-score nltk bert-score
import sys
sys.path.insert(0, '..')
import requests
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

API_BASE = 'http://localhost:8000'

# Reference QA pairs (query → ideal answer keywords/phrases)
qa_pairs = [
    {
        'query': 'What are the prerequisites for DS201?',
        'reference': 'DS201 Data Science Fundamentals requires CS101 Introduction to Programming and STAT101 Statistics for Engineers as prerequisites.',
        'session': 'eval_001',
    },
    {
        'query': 'When does course registration close for the odd semester?',
        'reference': 'Course registration for the odd semester closes on 2024-07-18. Late registration is allowed until 2024-07-25 with a Rs. 500 fine.',
        'session': 'eval_002',
    },
    {
        'query': 'What is the minimum attendance required to appear for exams?',
        'reference': 'Minimum attendance of 75% is mandatory in each course. Students falling below 75% will be debarred from end-semester exams.',
        'session': 'eval_003',
    },
    {
        'query': 'How much is the BTech tuition fee per semester for domestic students?',
        'reference': 'The tuition fee for BTech domestic students is Rs. 85,000 per semester. Total per semester with hostel is Rs. 1,55,000.',
        'session': 'eval_004',
    },
    {
        'query': 'What are library borrowing rules?',
        'reference': 'BTech and MSc students can borrow up to 4 books at a time. The loan period is 14 days for regular books and 3 days for reference books. Late fine is Rs. 2 per day per book.',
        'session': 'eval_005',
    },
]

print(f'{len(qa_pairs)} evaluation pairs loaded.')

In [ ]:
def get_response(session_id, query):
    try:
        r = requests.post(f'{API_BASE}/chat', json={'session_id': session_id, 'message': query}, timeout=90)
        return r.json().get('response', '')
    except:
        # Return the tool result via RAG endpoint as fallback
        try:
            r = requests.post(f'{API_BASE}/rag', json={'query': query, 'top_k': 3}, timeout=30)
            docs = r.json().get('results', [])
            return ' '.join(d['content'][:200] for d in docs)
        except:
            return ''

# Collect model responses
print('Collecting model responses...')
for pair in qa_pairs:
    pair['hypothesis'] = get_response(pair['session'], pair['query'])
    print(f'  [{pair["session"]}] Response length: {len(pair["hypothesis"])} chars')
print('Done.')

In [ ]:
smoother = SmoothingFunction().method4
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

results = []
for pair in qa_pairs:
    ref_tokens = pair['reference'].lower().split()
    hyp_tokens = pair['hypothesis'].lower().split()

    # BLEU-1
    bleu1 = sentence_bleu([ref_tokens], hyp_tokens, weights=(1,0,0,0), smoothing_function=smoother)
    # BLEU-4
    bleu4 = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother)
    # ROUGE-L
    rouge_scores = scorer.score(pair['reference'], pair['hypothesis'])
    rouge_l = rouge_scores['rougeL'].fmeasure

    results.append({
        'Query': pair['query'][:55] + '...',
        'BLEU-1': round(bleu1, 3),
        'BLEU-4': round(bleu4, 3),
        'ROUGE-L': round(rouge_l, 3),
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nMean BLEU-1:  {df['BLEU-1'].mean():.3f}")
print(f"Mean BLEU-4:  {df['BLEU-4'].mean():.3f}")
print(f"Mean ROUGE-L: {df['ROUGE-L'].mean():.3f}")

In [ ]:
# Optional: BERTScore (requires GPU or is slow on CPU)
try:
    from bert_score import score as bert_score
    refs = [p['reference'] for p in qa_pairs]
    hyps = [p['hypothesis'] for p in qa_pairs]
    P, R, F1 = bert_score(hyps, refs, lang='en', verbose=False)
    print(f'BERTScore — Precision: {P.mean():.3f} | Recall: {R.mean():.3f} | F1: {F1.mean():.3f}')
except ImportError:
    print('BERTScore not installed. Run: pip install bert-score')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(df))
ax.plot(x, df['BLEU-1'], 'o-', label='BLEU-1', color='steelblue')
ax.plot(x, df['BLEU-4'], 's-', label='BLEU-4', color='darkorange')
ax.plot(x, df['ROUGE-L'], '^-', label='ROUGE-L', color='green')
ax.set_xticks(x)
ax.set_xticklabels([f'Q{i+1}' for i in x])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('BLEU & ROUGE-L Scores per Query')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('bleu_rouge_results.png', dpi=150)
plt.show()